In [ ]:
# Step 1: load cleaned corpus from previous data preparation

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / "outputs"

CORPUS_PATH = OUTPUT_DIR / "analysis_corpus_file_level_v2.csv"

corpus = pd.read_csv(CORPUS_PATH)

print("Corpus shape:", corpus.shape)
print("\nColumns:")
print(corpus.columns.tolist())

corpus.head()

In [ ]:
# Step 2: inspect corpus status flags

status_cols = [
    "read_status",
    "shared_transcript",
    "partial_transcript",
    "interview_notes_no_audio",
    "transcript_without_datalist_record",
    "non_standard_text_type",
    "include_in_kwic_corpus_preliminary",
    "requires_review_before_final_corpus"
]

for col in status_cols:
    print(f"\n--- {col} ---")
    print(corpus[col].value_counts(dropna=False))

In [ ]:
# Step 2.1: inspect all records flagged for review or special handling

special_records = corpus[
    (corpus["shared_transcript"] == True)
    | (corpus["partial_transcript"] == True)
    | (corpus["interview_notes_no_audio"] == True)
    | (corpus["transcript_without_datalist_record"] == True)
    | (corpus["non_standard_text_type"] == True)
    | (corpus["requires_review_before_final_corpus"] == True)
][[
    "file_name",
    "participant_ids_str",
    "shared_transcript",
    "partial_transcript",
    "interview_notes_no_audio",
    "transcript_without_datalist_record",
    "non_standard_text_type",
    "include_in_kwic_corpus_preliminary",
    "requires_review_before_final_corpus",
    "corpus_note"
]]

special_records

In [ ]:
# Step 3: define final corpus inclusion and exclusion rules

corpus["include_final"] = True
corpus["exclusion_reason"] = ""

# Exclude shared transcripts
mask_shared = corpus["shared_transcript"] == True
corpus.loc[mask_shared, "include_final"] = False
corpus.loc[mask_shared, "exclusion_reason"] = "shared_transcript"

# Exclude interview notes / no-audio record
mask_notes = corpus["interview_notes_no_audio"] == True
corpus.loc[mask_notes, "include_final"] = False
corpus.loc[mask_notes, "exclusion_reason"] = "interview_notes_no_audio"

# Exclude transcript without matching DataList record
mask_unmatched = corpus["transcript_without_datalist_record"] == True
corpus.loc[mask_unmatched, "include_final"] = False
corpus.loc[mask_unmatched, "exclusion_reason"] = "no_matching_datalist_record"

# Mark analysis status
corpus["analysis_status"] = "standard"

corpus.loc[
    corpus["partial_transcript"] == True,
    "analysis_status"
] = "partial"

corpus.loc[
    corpus["include_final"] == False,
    "analysis_status"
] = "excluded"

print("Final included files:", corpus["include_final"].sum())

print("\nAnalysis status:")
print(corpus["analysis_status"].value_counts())

print("\nExcluded files:")
display(
    corpus.loc[
        corpus["include_final"] == False,
        [
            "file_name",
            "participant_ids_str",
            "exclusion_reason"
        ]
    ]
)

In [ ]:
# Step 4: identify mixed-format transcripts for targeted preprocessing

mixed_ids = [
    "JUST030",
    "JUST076",
    "JUST092",
    "JUST101",
    "JUST321"
]

mixed_records = corpus[
    corpus["participant_ids_str"].isin(mixed_ids)
][[
    "file_name",
    "participant_ids_str",
    "non_standard_text_type",
    "text_clean"
]].copy()

print("Number of mixed-format records found:", len(mixed_records))

display(
    mixed_records[
        ["file_name", "participant_ids_str", "non_standard_text_type"]
    ]
)

In [ ]:
# Step 4.1: create paragraph-only text for mixed-format transcripts

from docx import Document
import re

def clean_analysis_text(text):
    """
    Light cleaning for final analytical text.
    Keeps wording and punctuation while normalising whitespace.
    """
    text = str(text)
    text = text.replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def extract_paragraph_only_text(file_path):
    """
    Extract only normal Word paragraphs.
    Text stored inside Word tables is deliberately excluded.
    """
    doc = Document(file_path)

    paragraphs = [
        p.text.strip()
        for p in doc.paragraphs
        if p.text.strip()
    ]

    text = "\n".join(paragraphs)

    return clean_analysis_text(text)


# Start from the existing cleaned text for all records
corpus["analysis_text_final"] = corpus["text_clean"]

# Replace only the five mixed-format transcripts
mixed_mask = corpus["participant_ids_str"].isin(mixed_ids)

for idx in corpus.index[mixed_mask]:
    file_path = corpus.loc[idx, "file_path"]

    corpus.loc[idx, "analysis_text_final"] = (
        extract_paragraph_only_text(file_path)
    )

print("Mixed-format transcripts updated:",
      mixed_mask.sum())

In [ ]:
# Step 4.2: compare original and final analytical text lengths

mixed_check = corpus.loc[
    mixed_mask,
    [
        "file_name",
        "participant_ids_str",
        "text_clean",
        "analysis_text_final"
    ]
].copy()

mixed_check["original_word_count"] = (
    mixed_check["text_clean"]
    .fillna("")
    .str.split()
    .str.len()
)

mixed_check["final_word_count"] = (
    mixed_check["analysis_text_final"]
    .fillna("")
    .str.split()
    .str.len()
)

mixed_check["words_removed"] = (
    mixed_check["original_word_count"]
    - mixed_check["final_word_count"]
)

mixed_check["proportion_retained"] = (
    mixed_check["final_word_count"]
    / mixed_check["original_word_count"]
)

display(
    mixed_check[
        [
            "file_name",
            "participant_ids_str",
            "original_word_count",
            "final_word_count",
            "words_removed",
            "proportion_retained"
        ]
    ]
)

In [ ]:
# Step 4.3: inspect a mixed-format example after table removal

just076_final = corpus.loc[
    corpus["participant_ids_str"] == "JUST076",
    "analysis_text_final"
].iloc[0]

print(just076_final[:5000])

In [ ]:
# Step 5: save the final analytical corpus and corpus audit files

final_corpus = corpus.loc[
    corpus["include_final"] == True
].copy()

# Basic checks
print("Final analytical corpus size:", len(final_corpus))
print("Partial transcripts retained:",
      final_corpus["partial_transcript"].sum())

print("Missing analysis_text_final:",
      final_corpus["analysis_text_final"].isna().sum())

print("Empty analysis_text_final:",
      (final_corpus["analysis_text_final"].fillna("").str.strip() == "").sum())

In [ ]:
# Step 5.1: save final corpus outputs

FINAL_CORPUS_PATH = OUTPUT_DIR / "final_analysis_corpus_v1.csv"
FINAL_MANIFEST_PATH = OUTPUT_DIR / "final_corpus_manifest_v1.csv"

# Full analytical corpus: includes text for later KWIC
final_corpus.to_csv(
    FINAL_CORPUS_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Manifest: lighter audit table without full transcript text
manifest_cols = [
    "file_name",
    "participant_ids_str",
    "analysis_status",
    "include_final",
    "exclusion_reason",
    "partial_transcript",
    "word_count"
]

final_manifest = corpus[manifest_cols].copy()

final_manifest.to_csv(
    FINAL_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(FINAL_CORPUS_PATH)
print(FINAL_MANIFEST_PATH)

In [ ]:
# Step 5.2: verify saved final corpus

check_final = pd.read_csv(FINAL_CORPUS_PATH)

print("Reloaded final corpus shape:", check_final.shape)
print("Unique files:", check_final["file_name"].nunique())
print("Unique participant ID strings:",
      check_final["participant_ids_str"].nunique())

In [ ]:
# Step 6: load the revised institution dictionary used in the pilot

CODE_DIR = PROJECT_DIR / "my code"

DICTIONARY_PATH = CODE_DIR / "institution_dictionary_v2.xlsx"

dictionary = pd.read_excel(DICTIONARY_PATH)

print("Dictionary shape:", dictionary.shape)
print("\nColumns:")
print(dictionary.columns.tolist())

display(dictionary.head(10))

In [ ]:
# Step 7: inspect dictionary category and include/exclude distributions

print("Category counts:")
print(dictionary["category"].value_counts())

print("\nInclude / exclude counts:")
print(dictionary["include/exclude"].value_counts())

print("\nIncluded keywords by category:")
included_dictionary = dictionary[
    dictionary["include/exclude"].str.lower() == "include"
].copy()

display(
    included_dictionary.groupby("category")
    .size()
    .reset_index(name="n_included_keywords")
)

In [ ]:
# Step 8: load the human-reviewed pilot annotation

PILOT_PATH = OUTPUT_DIR / "annotation_pilot_sample_v1_human_reviewed.xlsx"

pilot = pd.read_excel(PILOT_PATH)

print("Pilot shape:", pilot.shape)
print("\nColumns:")
print(pilot.columns.tolist())

display(pilot.head())

In [ ]:
# Step 8.1: summarise pilot validity by keyword

pilot["valid_mention"] = (
    pilot["valid_mention"]
    .astype(str)
    .str.strip()
    .str.lower()
)

keyword_validity = (
    pilot
    .groupby(["category", "keyword"])
    .agg(
        pilot_n=("valid_mention", "size"),
        valid_n=("valid_mention", lambda x: (x == "yes").sum()),
        invalid_n=("valid_mention", lambda x: (x == "no").sum())
    )
    .reset_index()
)

keyword_validity["validity_rate"] = (
    keyword_validity["valid_n"] / keyword_validity["pilot_n"]
)

keyword_validity = keyword_validity.sort_values(
    ["validity_rate", "pilot_n"],
    ascending=[True, False]
)

display(keyword_validity)

In [ ]:
# Step 8.2: summarise pilot validity by institution category

category_validity = (
    pilot
    .groupby("category")
    .agg(
        pilot_n=("valid_mention", "size"),
        valid_n=("valid_mention", lambda x: (x == "yes").sum()),
        invalid_n=("valid_mention", lambda x: (x == "no").sum())
    )
    .reset_index()
)

category_validity["validity_rate"] = (
    category_validity["valid_n"] / category_validity["pilot_n"]
)

category_validity = category_validity.sort_values(
    "validity_rate"
)

display(category_validity)

### Pilot-based dictionary review

The pilot annotation showed substantial differences in validity across institution categories. Police, legal and support-sector keywords produced high proportions of valid institutional mentions, while health and welfare/state keywords generated more false positives.

At keyword level, broad terms such as *nurse*, *doctor*, *housing*, *council*, *benefits* and *social workers* were more likely to occur in contexts that did not describe a substantive institutional role. However, the number of pilot observations for many individual keywords was small, so these results are not used to remove keywords or institution categories at this stage.

Instead, the pilot is treated as a diagnostic step. All five candidate institution categories are retained for full-corpus validation. Final decisions about whether health or welfare/state should enter detailed comparative analysis will be based on full-corpus volume, validity rate and distribution across unique transcripts.

In [ ]:
# Step 9: save candidate dictionary for full-corpus validation

candidate_dictionary = dictionary.copy()

candidate_dictionary["dictionary_stage"] = "candidate_full_corpus_validation"

CANDIDATE_DICT_PATH = (
    OUTPUT_DIR / "institution_dictionary_candidate_final_v1.xlsx"
)

candidate_dictionary.to_excel(
    CANDIDATE_DICT_PATH,
    index=False
)

print("Saved candidate dictionary:")
print(CANDIDATE_DICT_PATH)

print("\nDictionary summary:")
print(
    candidate_dictionary["include/exclude"]
    .value_counts()
)

In [ ]:
# Inspect keywords excluded from the final dictionary

dictionary[
    dictionary["include/exclude"].str.lower() == "exclude"
][["category", "keyword", "reason", "ambiguity_note"]]